# Figure 4: compare PPO checkpoints from 1k to 25k steps

- Finds every 1k–25k checkpoint for Paper, Normalized, Fract, and Mesh PPO.
- Evaluates each checkpoint deterministically over 10 rounds with 10 parallel simulations and tumour IDs 1–10.
- Uses 24-hour treatment availability with appointment delays disabled.
- Saves the evaluation table and a learning curve figure with one panel per PPO model.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))
PROJECT_DIR = next((candidate.resolve() for parent in (Path.cwd(), *Path.cwd().parents) for candidate in (parent, parent / "report_files") if (candidate / "train_normalized_PPO.py").is_file() and (candidate / "TumorGrowthEnv/tumor_growth/__init__.py").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not locate report_files")
for local_dir in (PROJECT_DIR, PROJECT_DIR / "TumorGrowthEnv"):
    if str(local_dir) not in sys.path:
        sys.path.insert(0, str(local_dir))

import gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from stable_baselines3 import PPO

import tumor_growth

PARALLEL_RUNS = 10
ROUNDS = 10
TUMOUR_IDS = list(range(1, 11))
DETERMINISTIC = True
RESULTS_DIR = PROJECT_DIR / "baselines/results"
MAX_TRAINING_STEPS = 25_000
EXPECTED_STEPS = set(range(1_000, MAX_TRAINING_STEPS + 1_000, 1_000))

In [ ]:
MODEL_DIRS = {
    "paper": RESULTS_DIR / "models/paper_ppo",
    "normalized": RESULTS_DIR / "models/normalized_ppo",
    "fract": RESULTS_DIR / "models/fract_ppo",
    "mesh": RESULTS_DIR / "models/mesh_ppo",
}

ENVIRONMENT_IDS = {
    "paper": "tumor_growth:tumor-growth-paper-ppo-v0",
    "normalized": "tumor_growth:tumor-growth-normalized-ppo-v0",
    "fract": "tumor_growth:tumor-growth-fract-ppo-v0",
    "mesh": "tumor_growth:tumor-growth-mesh-ppo-v0",
}

MODEL_LABELS = {
    "fract": "Fract PPO",
    "mesh": "Mesh PPO",
    "normalized": "Normalized PPO",
    "paper": "Paper PPO",
}

def available_checkpoints(model_type):
    model_dir = MODEL_DIRS[model_type]
    if not model_dir.is_dir():
        raise FileNotFoundError(model_dir)
    checkpoints = {}
    for path in model_dir.rglob("ppo_*_steps.zip"):
        step_text = path.stem.removeprefix("ppo_").removesuffix("_steps")
        if not step_text.isdigit():
            continue
        step = int(step_text)
        if step > MAX_TRAINING_STEPS:
            continue
        if step in checkpoints:
            raise RuntimeError(f"Duplicate checkpoint for {step} steps: {checkpoints[step]} and {path}")
        checkpoints[step] = path
    missing_steps = sorted(EXPECTED_STEPS - set(checkpoints))
    if missing_steps:
        missing = ", ".join(f"{step // 1000}k" for step in missing_steps)
        raise FileNotFoundError(f"Missing {model_type} checkpoints: {missing}")
    return dict(sorted(checkpoints.items()))

CHECKPOINTS = {
    model_type: available_checkpoints(model_type)
    for model_type in MODEL_LABELS
}
for model_type, checkpoints in CHECKPOINTS.items():
    print(f"Found {len(checkpoints)} checkpoints for {MODEL_LABELS[model_type]}")

In [ ]:
results = []

for model_type, checkpoints in CHECKPOINTS.items():
    for step, path in checkpoints.items():
        print(f"Evaluating {MODEL_LABELS[model_type]} at {step // 1000}k steps")
        model = PPO.load(path, device="cpu")
        env = gym.make(
            ENVIRONMENT_IDS[model_type],
            tumors_list=TUMOUR_IDS,
            parallel_runs=PARALLEL_RUNS,
            with_time=True,
            dosing_hours=24,
            appointment_delays=False,
        )

        outcomes = []
        try:
            for _ in range(ROUNDS):
                observation = env.reset()
                if isinstance(observation, tuple):
                    observation = observation[0]

                done = False
                while not done:
                    action, _ = model.predict(observation, deterministic=DETERMINISTIC)
                    result = env.step(action)
                    if len(result) == 4:
                        observation, _, done, _ = result
                    else:
                        observation, _, terminated, truncated, _ = result
                        done = terminated or truncated

                outcomes.extend(
                    np.asarray(env.unwrapped.tumor_cells, dtype=np.float64).ravel()
                )
        finally:
            env.close()

        outcomes = np.asarray(outcomes, dtype=np.float64)
        expected_outcomes = ROUNDS * PARALLEL_RUNS * len(TUMOUR_IDS)
        if outcomes.size != expected_outcomes:
            raise RuntimeError(
                f"Expected {expected_outcomes} outcomes for {model_type} at {step}, "
                f"received {outcomes.size}"
            )
        results.append({
            "model": MODEL_LABELS[model_type],
            "model_type": model_type,
            "training_steps": step,
            "training_steps_k": step // 1000,
            "mean_leftover_cells": outcomes.mean(),
            "std_leftover_cells": outcomes.std(ddof=1),
            "outcomes": outcomes.size,
            "rounds": ROUNDS,
            "parallel_runs": PARALLEL_RUNS,
            "tumour_ids": len(TUMOUR_IDS),
            "deterministic": DETERMINISTIC,
        })

results_df = pd.DataFrame(results).sort_values(["model_type", "training_steps"])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
csv_path = RESULTS_DIR / "fig04_checkpoint_evaluation_paper_normalized_fract_mesh_1k_to_25k_10x10.csv"
results_df.to_csv(csv_path, index=False)
print(f"Saved evaluation data to {csv_path}")
results_df.round(3)

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharex=True)
axes = axes.ravel()

for ax, (model_type, model_label) in zip(axes, MODEL_LABELS.items()):
    curve = results_df[results_df["model_type"].eq(model_type)].sort_values("training_steps")
    x = curve["training_steps_k"].to_numpy(dtype=float)
    mean = curve["mean_leftover_cells"].to_numpy(dtype=float)
    std = curve["std_leftover_cells"].to_numpy(dtype=float)
    ax.plot(x, mean, marker="o", markersize=4, linewidth=2.2)
    ax.fill_between(x, mean - std, mean + std, alpha=0.2, label="±1 standard deviation")
    ax.set_title(f"{model_label} performance from 1k to 25k steps")
    ax.set_xlabel("Training steps (thousands)")
    ax.set_ylabel("Leftover tumour cells")
    ax.set_xticks(list(range(1, 26, 2)))
    ax.legend()

fig.suptitle("Deterministic PPO checkpoint comparison", fontsize=16, fontweight="bold")
fig.tight_layout()
plot_path = RESULTS_DIR / "fig04_ppo_learning_curves_1k_to_25k.png"
fig.savefig(plot_path, dpi=300, bbox_inches="tight")
print(f"Saved Figure 4 to {plot_path}")